# 02 — Model training, comparison and Bayesian optimisation

Runs the same code as the CLI:

```
python -m src.models.train
python -m src.models.optimize
```

80 training trips → GroupKFold CV by trip; 10 student-testing trips → comparison. Evaluator trips are never read.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, IFrame, display
from src.config import load_config, resolve
cfg = load_config()
RESULTS = resolve(cfg["paths"]["results_dir"])

In [ ]:
RUN = False  # set True to (re)train from the notebook; otherwise show saved results
if RUN:
    from src.models import train, optimize
    train.main([])
    optimize.main([])

In [ ]:
comp = pd.read_csv(RESULTS / "model_comparison.csv")
cols = ["model", "family", "key_config", "cv_macro_f1_mean", "cv_macro_f1_std", "cv_top1_mean", "student_top1",
        "student_route_continuity", "student_hmm_top1", "student_macro_f1", "student_log_loss",
        "inference_ms_per_1k_pairs", "train_time_s", "train_top1"]
display(comp[cols].sort_values("cv_top1_mean", ascending=False))

In [ ]:
import json
print(json.dumps(json.loads((RESULTS / "model_selection.json").read_text()), indent=2))

In [ ]:
for name in ["cv_macro_f1", "cv_top1", "cv_top1_folds", "student_top1_macro_f1", "student_log_loss",
             "classifier_vs_hmm_top1", "route_continuity", "inference_time"]:
    p = RESULTS / "plots" / f"{name}.png"
    if p.exists(): display(Image(filename=str(p)))

## Per-model diagnostics (confusion matrix, feature importance, loss curves where they exist)

In [ ]:
for d in sorted((RESULTS / "models").iterdir()):
    for f in ["confusion_matrix.png", "feature_importance.png", "loss_curve.png", "validation_curve.png"]:
        if (d / f).exists():
            print(d.name, f); display(Image(filename=str(d / f)))

## Bayesian optimisation

In [ ]:
for d in (RESULTS / "optuna").iterdir():
    print(json.dumps(json.loads((d / "study_summary.json").read_text()), indent=2))
    display(pd.read_csv(d / "trials.csv").sort_values("value", ascending=False).head(10))
    for f in ["optimization_history.png", "param_importances.png"]:
        if (d / f).exists(): display(Image(filename=str(d / f)))
display(pd.read_csv(RESULTS / "tuned_vs_untuned.csv"))